# Semana 18 · ¿Cómo se viaja de un planeta a otro?

Este cuaderno acompaña a la sesión de proyecto de la semana 18: sistemas de ecuaciones con NumPy y el cálculo completo de una transferencia de Hohmann a Marte.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Sistemas de ecuaciones con NumPy

Un sistema **lineal** (incógnitas sin cuadrados ni productos entre ellas) se escribe como una tabla de coeficientes $A$ y una lista de términos independientes $b$. Para $2x + 3y = 12$ y $x - y = 1$:

$$A = \begin{pmatrix} 2 & 3 \\ 1 & -1 \end{pmatrix}, \qquad b = \begin{pmatrix} 12 \\ 1 \end{pmatrix}.$$

`np.linalg.solve(A, b)` devuelve las incógnitas. (Esa tabla se llama **matriz**; la estudiarás la semana que viene.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[2.0, 3.0],
              [1.0, -1.0]])
b = np.array([12.0, 1.0])
print("x, y =", np.linalg.solve(A, b))

## 2. Las velocidades en el perihelio y el afelio

Las dos ecuaciones de la sesión 2 no son lineales en $v_\text{p}$ y $v_\text{a}$, pero hay un truco: si elevamos al cuadrado la primera, las dos son lineales en $X = v_\text{p}^2$ e $Y = v_\text{a}^2$:

$$r_\text{p}^2\,X - r_\text{a}^2\,Y = 0, \qquad \tfrac{1}{2}X - \tfrac{1}{2}Y = \frac{GM}{r_\text{p}} - \frac{GM}{r_\text{a}}.$$

Lo resolvemos para la órbita de transferencia de la Tierra a Marte, alrededor del Sol.

In [ ]:
GM_sol = 1.327e20          # m³/s²
ua = 1.496e11              # m
r1, r2 = 1.000 * ua, 1.524 * ua      # radios de las órbitas de la Tierra y Marte (circulares)

A = np.array([[r1**2, -r2**2],
              [0.5,   -0.5]])
b = np.array([0.0, GM_sol / r1 - GM_sol / r2])
X, Y = np.linalg.solve(A, b)
vp, va = np.sqrt(X), np.sqrt(Y)
print(f"Sistema:   v perihelio = {vp/1e3:.2f} km/s   v afelio = {va/1e3:.2f} km/s")

def vis_viva(r, a, GM=GM_sol):
    return np.sqrt(GM * (2 / r - 1 / a))

a_t = (r1 + r2) / 2
print(f"Vis viva:  v perihelio = {vis_viva(r1, a_t)/1e3:.2f} km/s   v afelio = {vis_viva(r2, a_t)/1e3:.2f} km/s")

## 3. La transferencia de Hohmann a cada planeta

Para cada destino calculamos:

- la velocidad circular de la Tierra y del planeta, $\sqrt{GM/r}$;
- las velocidades de la elipse de transferencia en sus dos extremos (vis viva);
- los dos impulsos, $\Delta v_1$ al salir y $\Delta v_2$ al llegar (en valor absoluto: para ir hacia dentro, a Venus o Mercurio, hay que frenar);
- la duración, medio periodo de la elipse;
- el ángulo con el que el planeta debe ir por delante de la Tierra al salir.

In [ ]:
año = 365.25 * 86400
destinos = {"Mercurio": 0.387, "Venus": 0.723, "Marte": 1.524, "Júpiter": 5.203, "Saturno": 9.537}

def hohmann(r1, r2, GM=GM_sol):
    a_t = (r1 + r2) / 2
    v1c, v2c = np.sqrt(GM / r1), np.sqrt(GM / r2)
    v1t, v2t = vis_viva(r1, a_t, GM), vis_viva(r2, a_t, GM)
    t = np.pi * np.sqrt(a_t**3 / GM)
    return abs(v1t - v1c), abs(v2c - v2t), t

for nombre, r_ua in destinos.items():
    dv1, dv2, t = hohmann(1.0 * ua, r_ua * ua)
    T_planeta = r_ua ** 1.5 * año                     # tercera ley de Kepler
    avance = 360 * t / T_planeta                      # grados que avanza el planeta durante el viaje
    fase = (180 - avance) % 360
    print(f"{nombre:9s} viaje {t/86400:7.0f} días ({t/año:5.2f} años)   Δv1 = {dv1/1e3:5.2f} km/s   Δv2 = {dv2/1e3:5.2f} km/s   ángulo de salida = {fase:6.1f}°")

El ángulo se mide en el sentido del movimiento, entre 0° y 360°. Para Venus sale 306°, que equivale a ir 54° **por detrás** de la Tierra: como Venus es más rápido, la alcanza durante el viaje. Mercurio da más de una vuelta completa mientras la nave viaja, y el cálculo descarta esas vueltas enteras con el operador `%` (resto de la división entre 360).

**Ejercicio.** Calcula el impulso desde una órbita baja alrededor de la Tierra (400 km) para salir hacia Marte. La nave debe dejar la Tierra con una velocidad sobrante $v_\infty = \Delta v_1$; por conservación de la energía, en la órbita baja necesita $v = \sqrt{v_\infty^2 + v_\text{esc}^2}$, y ya lleva la velocidad circular de esa órbita.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
GM_tierra = 3.986e14
r_leo = 6.371e6 + 400e3
dv1, _, _ = hohmann(1.0 * ua, 1.524 * ua)
v_esc = np.sqrt(2 * GM_tierra / r_leo)
v_circ = np.sqrt(GM_tierra / r_leo)
v_necesaria = np.sqrt(dv1**2 + v_esc**2)
print(f"v_inf = {dv1/1e3:.2f} km/s, v_esc = {v_esc/1e3:.2f} km/s, v necesaria = {v_necesaria/1e3:.2f} km/s")
print(f"Impulso desde la órbita baja: {(v_necesaria - v_circ)/1e3:.2f} km/s")

## 4. El viaje, paso a paso

Colocamos la Tierra en el ángulo 0 el día de salida y Marte, 44° por delante. Los planetas giran en órbitas circulares (semana 13). La nave recorre la elipse de transferencia, y su posición en cada momento la da la ecuación de Kepler (semana 14), con el perihelio en el punto de salida.

In [ ]:
def resolver_kepler(M, e):
    E = np.array(M, dtype=float)
    for _ in range(50):
        E = E - (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
    return E

r1_ua, r2_ua = 1.0, 1.524
a_t = (r1_ua + r2_ua) / 2
e_t = (r2_ua - r1_ua) / (r2_ua + r1_ua)
T_t = a_t ** 1.5 * 365.25                  # periodo de la elipse de transferencia, en días
t_viaje = T_t / 2
T_T, T_M = 365.25, 686.98
fase = np.radians(180 - 360 * t_viaje / T_M)

def tierra(t): return r1_ua * np.cos(2*np.pi*t/T_T), r1_ua * np.sin(2*np.pi*t/T_T)
def marte(t):  return r2_ua * np.cos(fase + 2*np.pi*t/T_M), r2_ua * np.sin(fase + 2*np.pi*t/T_M)
def nave(t):
    E = resolver_kepler(2 * np.pi * t / T_t, e_t)
    return a_t * (np.cos(E) - e_t), a_t * np.sqrt(1 - e_t**2) * np.sin(E)

print(f"Duración del viaje: {t_viaje:.0f} días   ángulo de salida: {np.degrees(fase):.1f}°")

ang = np.linspace(0, 2*np.pi, 300)
dias = np.linspace(0, t_viaje, 5)
fig, axs = plt.subplots(1, 5, figsize=(16, 3.6))
for ax, d in zip(axs, dias):
    ax.plot(r1_ua*np.cos(ang), r1_ua*np.sin(ang), color="lightgray")
    ax.plot(r2_ua*np.cos(ang), r2_ua*np.sin(ang), color="lightgray")
    tt = np.linspace(0, d, 100)
    ax.plot(*nave(tt), color="k", lw=1)
    ax.plot(0, 0, "o", color="orange", ms=8)
    ax.plot(*tierra(d), "o", color="tab:blue", label="Tierra")
    ax.plot(*marte(d), "o", color="tab:red", label="Marte")
    ax.plot(*nave(np.array([d])), "^", color="k", label="nave")
    ax.set_title(f"día {d:.0f}"); ax.set_aspect("equal"); ax.set_xlim(-1.7, 1.7); ax.set_ylim(-1.7, 1.7); ax.axis("off")
axs[0].legend(loc="lower left", fontsize=7)
plt.tight_layout(); plt.show()

xm, ym = marte(t_viaje); xn, yn = nave(np.array([t_viaje]))
print(f"Distancia nave-Marte a la llegada: {np.hypot(xm - xn[0], ym - yn[0]):.4f} ua")

## 5. Dónde se ve Marte el día de salida

Desde la Tierra, el ángulo entre el Sol y Marte en el cielo (la **elongación**) es el ángulo entre los vectores que van de la Tierra al Sol y de la Tierra a Marte. Se calcula con el **producto escalar**: para dos vectores $\vec{u}$ y $\vec{w}$, $\cos\alpha = (u_x w_x + u_y w_y)/(|\vec{u}|\,|\vec{w}|)$. En NumPy, `u @ w` calcula $u_x w_x + u_y w_y$.

**Ejercicio.** Calcula la elongación de Marte el día de salida. ¿Se verá por la tarde, a medianoche o de madrugada?

In [ ]:
# Tu código aquí

In [ ]:
# Solución
T0 = np.array(tierra(0.0)); M0 = np.array(marte(0.0))
u = -T0               # de la Tierra al Sol (el Sol está en el origen)
w = M0 - T0           # de la Tierra a Marte
elong = np.degrees(np.arccos(u @ w / (np.linalg.norm(u) * np.linalg.norm(w))))
print(f"Elongación de Marte el día de salida: {elong:.0f}°")

Unos 95°: Marte está a un cuarto de cielo del Sol, por el lado oeste (va por delante de la Tierra en su órbita), y se ve de madrugada. En el cielo real de principios de noviembre de 2026 la elongación es algo menor, unos 80°, porque las órbitas no son circulares y la fecha no coincide exactamente con la salida ideal; lo calcularás con precisión en la semana 24.

## 6. Experimenta

1. Calcula la transferencia de la Tierra a Venus y dibújala igual que la de Marte.
2. Desde la órbita baja terrestre, ¿qué impulso hace falta para llegar a la órbita geoestacionaria con una transferencia de Hohmann? Usa `hohmann` con $GM_\text{T}$ y radios de 6771 y 42 164 km.

In [ ]:
# Tu experimento aquí